# Prism quickstart: tracing a wrong total back to the fields that caused it

A Spark job computes each flight's duration with a user-defined function (UDF) and sums the durations per departure hour. Some totals come out negative, which is physically impossible. This notebook uses Prism to find which flights produced the bad total, which of their fields the UDF read, and which flights moved the total the most.

Prism attaches to unmodified Spark through the `spark.sql.extensions` setting, which this container already sets. The notebook runs against the Spark cluster that Docker Compose starts, and it also runs in local mode when `SPARK_MASTER` is unset. The `PRISM_SCALE` setting multiplies the size of every table the notebooks generate.

In [1]:
import os
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from prism import Prism, udf

# one executor per worker, sized to the worker
spark = (SparkSession.builder.appName("prism-quickstart")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.executor.cores", os.environ.get("WORKER_CORES", "2"))
         .config("spark.executor.memory", os.environ.get("WORKER_MEMORY", "2g"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
DATA = os.environ.get("PRISM_DATA", "/data")
SCALE = float(os.environ.get("PRISM_SCALE", "1"))   # multiplies every generated table's rows
print("Spark", spark.version, "on", spark.sparkContext.master, "| data scale", SCALE)

Spark 4.1.2 on spark://spark-master:7077 | data scale 1.0


## 1. Generate the flights

Each flight has an id, a carrier, a departure hour, and departure and arrival times stored as minutes since midnight (0 to 1439). A flight that departs late in the evening and lands after midnight has an arrival minute smaller than its departure minute. That wrap-around is the input the bug mishandles.

In [2]:
n = int(200_000 * SCALE)
flights = (spark.range(0, n, numPartitions=8)
           .select(F.col("id").alias("flight"),
                   F.element_at(F.array(*[F.lit(c) for c in ["AA", "DL", "UA", "WN"]]),
                                (F.col("id") % 4 + 1).cast("int")).alias("carrier"),
                   (F.col("id") % 24).cast("int").alias("dep_hr"))
           .withColumn("dep_min", F.col("dep_hr") * 60 + F.col("flight") * 7 % 60)
           .withColumn("arr_min", (F.col("dep_min") + 30 + F.col("flight") % 120) % 1440))
flights.write.mode("overwrite").parquet(f"{DATA}/flights.parquet")
flights = spark.read.parquet(f"{DATA}/flights.parquet")
flights.show(3)

+------+-------+------+-------+-------+
|flight|carrier|dep_hr|dep_min|arr_min|
+------+-------+------+-------+-------+
| 50000|     AA|     8|    500|    610|
| 50001|     DL|     9|    567|    678|
| 50002|     UA|    10|    634|    746|
+------+-------+------+-------+-------+
only showing top 3 rows


## 2. Run the job once with Prism capturing lineage

The UDF subtracts the two minute fields, which is wrong for flights that cross midnight. Three calls turn Prism on before the job runs. `enable_capture` records which input records feed each output. `enable_udf_aware` records which fields the UDF read, and the `lazy-cell` mode defers the per-record field tracer until a trace asks for it, so the job pays almost nothing for it. `enable_influence` records how much each record contributed to its group's sum.

In [3]:
@udf(DoubleType())
def duration(arr_min, dep_min):
    return float(arr_min - dep_min)   # bug: a flight crossing midnight should add 1440

p = Prism(spark)
p.enable_capture()
p.enable_udf_aware(duration, mode="lazy-cell")
p.enable_influence()

result = (flights.withColumn("dur", duration("arr_min", "dep_min"))
          .groupBy("dep_hr").agg(F.sum("dur").alias("total")))
rows = p.collect_with_lineage(result)          # one run of the job, lineage captured in passing
for row, _ in sorted(rows, key=lambda r: r[0]["dep_hr"])[-4:]:
    print(row)

Row(dep_hr=20, total=816562.0)
Row(dep_hr=21, total=-1574145.0)
Row(dep_hr=22, total=-3964852.0)
Row(dep_hr=23, total=-11157959.0)


## 3. Trace the most negative total back to its source flights

`trace` starts from an output row and `go_back` walks the lineage one stage at a time until it reaches the scan. The trace returns exactly the flights of that hour, and nothing else.

In [4]:
worst_row, worst_id = min(rows, key=lambda r: r[0]["total"])
print("suspicious output:", worst_row)

cur = p.trace(result, [worst_id])
while not cur.at_scan:
    cur = cur.go_back()
witnesses = cur.show()
expected = flights.filter(F.col("dep_hr") == worst_row["dep_hr"]).count()
print(len(witnesses), "flights traced, out of", n, "| flights in that hour:", expected)
assert len(witnesses) == expected

suspicious output: Row(dep_hr=23, total=-11157959.0)


8333 flights traced, out of 200000 | flights in that hour: 8333


## 4. Field-level lineage

Record-level provenance would stop here and hand back whole rows. Prism also reports, for each traced record, the fields the UDF computed its value from (`_data`) and the fields that only decided a branch (`_control`). The duration has no branch, so the control set is empty, and the data set names two of the five columns. The flight id, the carrier, and the hour are ruled out before a single value is inspected.

In [5]:
w = witnesses[0]
print("data fields:   ", w["_data"])
print("control fields:", w["_control"])
assert w["_data"] == ["arr_min", "dep_min"] and w["_control"] == []

data fields:    ['arr_min', 'dep_min']
control fields: []


## 5. Rank the flights by influence

The influence of a record on a sum is the amount the sum would change if that record were removed, which for a sum is the record's own value. Prism reads the aggregate from the query plan and computes this in one pass, with no re-execution. The midnight-crossing flights rank first, each with a duration near minus 1400 minutes.

In [6]:
influence = p.trace(result, [worst_id]).influence("total")
for record_id, value in influence.top(5):
    print(f"record {record_id}: {value:8.1f} minutes")
assert influence.top(1)[0][1] < -1000

record 63:  -1387.0 minutes
record 183:  -1387.0 minutes
record 303:  -1387.0 minutes
record 423:  -1387.0 minutes
record 543:  -1387.0 minutes


## 6. Assertion-directed capture

When the developer can name the suspicious group by its key before the run, for example `dep_hr = 23`, Prism pushes that assertion into the capture and records lineage for that group alone. The trace is the same, and the stored lineage shrinks by the share of the data outside the group.

In [7]:
eager_mb = sum(p.lineage_size(result)) / 1e6
p.enable_focus(f"dep_hr = {worst_row['dep_hr']}")
focused = (flights.withColumn("dur", duration("arr_min", "dep_min"))
           .groupBy("dep_hr").agg(F.sum("dur").alias("total")))
p.collect_with_lineage(focused)
focused_mb = sum(p.lineage_size(focused)) / 1e6
p.disable_focus()
print(f"eager lineage: {eager_mb:.3f} MB   assertion-directed: {focused_mb:.3f} MB")
assert focused_mb < eager_mb

eager lineage: 0.817 MB   assertion-directed: 0.035 MB


In [8]:
spark.stop()